In [1]:
import os

In [2]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\notebooks'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow'

In [ ]:
## Preparing entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    TRANSFORMATION_REPORT: Path

In [6]:
## Configuration

from wdmproject.constants import *
from wdmproject.utils.common import read_yaml, create_directories

In [8]:
## Configuration manager class
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])

    
    def get_data_tranformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            TRANSFORMATION_REPORT=Path(config.TRANSFORMATION_REPORT)
        )

        return data_transformation_config

In [ ]:
## Component

import os
import json
from wdmproject import logger
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pickle



In [ ]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.data = None
        self.transformation_report = {}


    ## Load Dataset for transformation
    def load_data(self):
        try:
            self.data = pd.read_excel(self.config.unzip_data_dir)
            with open(self.config.STATUS_FILE, 'w') as f:
                    f.write(f"Data Loaded Successfully. \n")
            logger.info('Data Loaded Succesfully.')

            self.transformation_report['dataset_load'] = {
                 "status" : "Success",
                 "rows" : self.data.shape[0],
                 "columns" : self.data.shape[1]
            }


        except Exception as e:
            logger.error(f"Error Loading Dataset. {e}")
            self.transformation_report['dataset_load'] = {
                 "status" : "Failed",
                 "error_msg" : "Error Loading Dataset.",
                 "error" : str(e),
            }
            raise e
        

    
    
    
    ## Saving Transformation Report 
    def save_transformation_report(self):

        report_path = self.config.TRANSFORMATION_REPORT

        with open(report_path, "w") as f:

            json.dump(self.transformation_report, f, indent=4)

        logger.info(f"Transformation report saved at {report_path}")


    ## Initialising Data Transformation Pipeline 

    def initiate_data_transformation(self):
        try: 
            
            ## Intiating Data Transformation stage

            self.transformation_report["stage_metadata"] = {
                "stage_name" : "Data Transformation",
                "stage_status" : "Running",
                "start_time" : str(datetime.now())
            }

            logger.info("Initiating Data Transformation Stage.")

            ## Saving start_time
            start_time = datetime.now()

            ## Loading Dataset
            self.load_data()

            ## Temporary clean currency from columns for transformation checks
            self.clean_currency_columns()

           

            ## Stage Success
            self.transformation_report["stage_metadata"]["stage_status"] = "Success"
            self.transformation_report["stage_metadata"]["end_time"] = str(datetime.now())

            ## Calculating Stage Duration
             
            stage_duration = (datetime.now() - start_time).total_seconds()
            self.transformation_report["stage_metadata"]["stage_duration"] = stage_duration
            
            # Saving Transformations Report JSON
            self.save_transformation_report()

            logger.info("All Data Transformation Checks Completed Successfully.")

        except Exception as e:
            
            logger.error(f"Transformation Error : {e}")

            end_time = datetime.now()

            self.transformation_report["stage_metadata"]["stage_status"] = "Failed"
            self.transformation_report["stage_metadata"]["end_time"] = str(end_time)
            self.transformation_report["stage_metadata"]["error"] = str(e)

            self.save_transformation_report()

            raise e


        